<a href="https://colab.research.google.com/github/adas754/nlp/blob/main/bilstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
#pad_sequences to make all sentences the same length
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

In [23]:
df = pd.read_csv('/content/fake_news_dataset.csv')
df.head()
#df['label'].value_counts()
#df.info()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [24]:
# replace missing titles with empty strings
df['title'] = df['title'].fillna('')

# replace missing article text with empty strings
df['text'] = df['text'].fillna('')

# combine title and article text into one column
df['content'] = df['title'] + ' ' + df['text']

# convert fake label into 0 and real label into 1
df['label'] = df['label'].str.lower().map({'fake': 0,'real': 1})

# remove rows where the label could not be converted
df = df.dropna(subset=['label'])

X = df['content']

y = df['label'].astype(int)


In [25]:
X.head()

,content
0,Foreign Democrat final. more tax development b...
1,To offer down resource great point. probably g...
2,Himself church myself carry. them identify for...
3,You unit its should. phone which item yard Rep...
4,Billion believe employee summer how. wonder my...


In [26]:
y

,label
0,1
1,0
2,0
3,0
4,0
...,...
19995,0
19996,1
19997,1
19998,0


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
# create a tokenizer that keeps a maximum of 20,000 words
# oov_token handles words that are not in the vocabulary

tokenizer = Tokenizer(
    num_words=20000,
    oov_token='<OOV>'
)

# learn the word vocabulary using only training data
tokenizer.fit_on_texts(X_train)

# convert training text into numerical sequences
X_train = tokenizer.texts_to_sequences(X_train)

# convert testing text into numerical sequences
X_test = tokenizer.texts_to_sequences(X_test)



In [29]:
# set the maximum number of tokens in each article
max_length = 300

# Make every training article exactly 300 tokens long
# padding='post' adds zeros at the end
# truncating='post' removes extra tokens from the end

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# make every testing article exactly 300 tokens long

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)


In [32]:
# ceate an empty Sequential model
model = Sequential()

# Convert each word ID into a vector of 64 numbers
# 20000 = maximum vocabulary size
# 64 = embedding dimension

model.add(Embedding(20000, 64))

# Bidirectional LSTM reads the article in two directions
# Forward LSTM reads left to right
# Backward LSTM reads right to left
# 64 = number of LSTM units in each direction

model.add(Bidirectional(LSTM(64)))

# randomly switch off 30% of units during training
# its helps reduce overfitting

model.add(Dropout(0.3))

# dense layer with 32 neurons


model.add(Dense(32, activation='relu'))


model.add(Dense(1, activation='sigmoid'))


model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:

model.compile(


    optimizer='adam',

    loss='binary_crossentropy',


    metrics=['accuracy']
)

In [34]:

history = model.fit(


    X_train,


    y_train,


    epochs=10,

    # process 64 articles in one batch
    batch_size=64,

    # use 20% of training data for validation
    validation_split=0.2
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 0.5037 - loss: 0.6935 - val_accuracy: 0.4931 - val_loss: 0.6933
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.5186 - loss: 0.6916 - val_accuracy: 0.4900 - val_loss: 0.6939
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.5794 - loss: 0.6751 - val_accuracy: 0.4944 - val_loss: 0.7102
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.6187 - loss: 0.6543 - val_accuracy: 0.4803 - val_loss: 0.7302
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.6389 - loss: 0.6386 - val_accuracy: 0.4966 - val_loss: 0.7389
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.6645 - loss: 0.6196 - val_accuracy: 0.4941 - val_loss: 0.7610
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6755 - loss: 0.6008 - val_accuracy: 0.4963 - val_loss: 0.7763
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.6957 - loss: 0.5740 - val_acc

In [35]:
loss, accuracy = model.evaluate(X_test, y_test)


print("Test Loss:", loss)

print("Test Accuracy:", accuracy)

125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4972 - loss: 0.9383
Test Loss: 0.938322901725769
Test Accuracy: 0.4972499907016754


In [36]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (64, 300, 64)          │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (64, 128)              │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (64, 128)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (64, 32)               │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (64, 1)                │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,629 (15.45 MB)

 Trainable params: 1,350,209 (5.15 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,700,420 (10.30 MB)

In [37]:
y_probability = model.predict(X_test)

y_pred = (y_probability >= 0.5).astype(int).ravel()


print("Predictions:", y_pred[:10])

print("Actual labels:", y_test.values[:10])

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Predictions: [1 1 1 1 1 0 0 1 1 1]
Actual labels: [1 0 0 1 0 0 1 0 1 1]


In [38]:
print(classification_report(
    y_test,
    y_pred,
    target_names=['Fake', 'Real']
))

              precision    recall  f1-score   support

        Fake       0.50      0.61      0.55      2011
        Real       0.49      0.39      0.43      1989

    accuracy                           0.50      4000
   macro avg       0.50      0.50      0.49      4000
weighted avg       0.50      0.50      0.49      4000



In [39]:
sample_news = """
The government announced a new education policy
to improve schools and provide better facilities
for students.
"""

# convert the new article into numerical tokens
sample_sequence = tokenizer.texts_to_sequences([sample_news])

#  make the sequence 300 tokens long
sample_padded = pad_sequences(
    sample_sequence,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# predict whether the article is fake or real
prediction = model.predict(sample_padded)

# display the predicted probability
print("Prediction Probability:", prediction[0][0])

# if probability is 0.5 or higher, predict real news
if prediction[0][0] >= 0.5:
    print("REAL NEWS")

else:
    print("FAKE NEWS")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step
Prediction Probability: 0.3279869
FAKE NEWS


In [ ]:
model.save('bidirectional_lstm_fake_news.keras')


print("Model saved successfully!")

Model saved successfully!


In [40]:

sample_news = """
Scientists have confirmed that drinking ordinary
salt water every morning gives humans the ability
to fly. The government has secretly approved this
discovery and all citizens will soon receive
special flying powers. Experts say this is 100%
true, but no scientific evidence has been released.
"""

sample_sequence = tokenizer.texts_to_sequences([sample_news])


sample_padded = pad_sequences(
    sample_sequence,
    maxlen=max_length,
    padding='post',
    truncating='post'
)


prediction = model.predict(sample_padded)


print("Prediction Probability:", prediction[0][0])

if prediction[0][0] >= 0.5:
    print("REAL NEWS")
else:
    print("FAKE NEWS")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Prediction Probability: 0.4596486
FAKE NEWS
